# Epochs — Dataset Exploration

Inspect FMA Small metadata, target genres, artist information, and audio-path mapping.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
METADATA_DIR = PROJECT_ROOT / "data" / "extracted" / "fma_metadata"
AUDIO_DIR = PROJECT_ROOT / "data" / "extracted" / "fma_small"

tracks_path = METADATA_DIR / "tracks.csv"
genres_path = METADATA_DIR / "genres.csv"

tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1])
genres = pd.read_csv(genres_path, index_col=0)

print("tracks shape:", tracks.shape)
print("genres shape:", genres.shape)


tracks shape: (106574, 52)
genres shape: (163, 4)


In [2]:
tracks.head()


album                                                     \
         comments         date_created        date_released engineer   
track_id                                                               
2               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
3               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
5               0  2008-11-26 01:44:45  2009-01-05 00:00:00      NaN   
10              0  2008-11-26 01:45:08  2008-02-06 00:00:00      NaN   
20              0  2008-11-26 01:45:05  2009-01-06 00:00:00      NaN   

                                                                          \
         favorites id                                information listens   
track_id                                                                   
2                4  1                                    <p></p>    6073   
3                4  1                                    <p></p>    6073   
5                4  1                                    <p></p>    6073   
10               4  6                                        NaN   47632   
20               2  4  <p> "spiritual songs" from Nicky Cook</p>    2710   

                        ...       track                         \
         producer tags  ... information interest language_code   
track_id                ...                                      
2             NaN   []  ...         NaN     4656            en   
3             NaN   []  ...         NaN     1470            en   
5             NaN   []  ...         NaN     1933            en   
10            NaN   []  ...         NaN    54881            en   
20            NaN   []  ...         NaN      978            en   

                                                                              \
                                                    license listens lyricist   
track_id                                                                       
2         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1293      NaN   
3         Attribution-NonCommercial-ShareAlike 3.0 Inter...     514      NaN   
5         Attribution-NonCommercial-ShareAlike 3.0 Inter...    1151      NaN   
10        Attribution-NonCommercial-NoDerivatives (aka M...   50135      NaN   
20        Attribution-NonCommercial-NoDerivatives (aka M...     361      NaN   

                                                 
         number publisher tags            title  
track_id                                         
2             3       NaN   []             Food  
3             4       NaN   []     Electric Ave  
5             6       NaN   []       This World  
10            1       NaN   []          Freeway  
20            3       NaN   []  Spiritual Level  

[5 rows x 52 columns]

In [3]:
print("Top-level columns:", tracks.columns.get_level_values(0).unique().tolist())
for group in tracks.columns.get_level_values(0).unique():
    print(f"{group}: {tracks[group].columns.tolist()}")


Top-level columns: ['album', 'artist', 'set', 'track']
album: ['comments', 'date_created', 'date_released', 'engineer', 'favorites', 'id', 'information', 'listens', 'producer', 'tags', 'title', 'tracks', 'type']
artist: ['active_year_begin', 'active_year_end', 'associated_labels', 'bio', 'comments', 'date_created', 'favorites', 'id', 'latitude', 'location', 'longitude', 'members', 'name', 'related_projects', 'tags', 'website', 'wikipedia_page']
set: ['split', 'subset']
track: ['bit_rate', 'comments', 'composer', 'date_created', 'date_recorded', 'duration', 'favorites', 'genre_top', 'genres', 'genres_all', 'information', 'interest', 'language_code', 'license', 'listens', 'lyricist', 'number', 'publisher', 'tags', 'title']


In [7]:
# Required fields for our project
required = [
    ("set", "split"),
    ("set", "subset"),
    ("track", "genre_top"),
    ("artist", "id"),
]

missing = [column for column in required if column not in tracks.columns]

print("Missing required columns:", missing)

if missing:
    raise ValueError(f"Required metadata columns are missing: {missing}")


# ---------------------------------------------------------
# Keep ONLY FMA Small
# ---------------------------------------------------------

small_tracks = tracks[
    tracks[("set", "subset")].astype(str).str.lower() == "small"
].copy()

print("FMA Small metadata rows:", len(small_tracks))


# ---------------------------------------------------------
# Build our clean project metadata table
# ---------------------------------------------------------

track_meta = pd.DataFrame({
    "track_id": small_tracks.index.astype(int),
    "split": small_tracks[("set", "split")].astype(str),
    "genre": small_tracks[("track", "genre_top")],
    "artist_id": small_tracks[("artist", "id")],
})


print("\nFMA Small tracks by split:")
print(track_meta["split"].value_counts())

print("\nFMA Small tracks by genre:")
print(track_meta["genre"].value_counts(dropna=False))


# ---------------------------------------------------------
# Build actual FMA Small audio paths
# ---------------------------------------------------------

def fma_audio_path(track_id):
    track_id = int(track_id)

    return (
        AUDIO_DIR
        / f"{track_id // 1000:03d}"
        / f"{track_id:06d}.mp3"
    )


track_meta["audio_path"] = track_meta["track_id"].map(fma_audio_path)
track_meta["audio_exists"] = track_meta["audio_path"].map(Path.exists)


print("\nAudio files found:", int(track_meta["audio_exists"].sum()))
print("Audio files missing:", int((~track_meta["audio_exists"]).sum()))


# ---------------------------------------------------------
# Keep only tracks with usable audio + genre label
# ---------------------------------------------------------

track_meta = track_meta[
    track_meta["audio_exists"]
    & track_meta["genre"].notna()
].copy()


print("\nFinal usable tracks:", len(track_meta))

print("\nFinal genre distribution:")
print(track_meta["genre"].value_counts())


# ---------------------------------------------------------
# Save clean metadata
# ---------------------------------------------------------

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

metadata_out = processed_dir / "fma_track_metadata.csv"

track_meta.to_csv(metadata_out, index=False)

print(f"\nSaved: {metadata_out}")

Missing required columns: []
FMA Small metadata rows: 8000

FMA Small tracks by split:
split
training      6400
validation     800
test           800
Name: count, dtype: int64

FMA Small tracks by genre:
genre
Hip-Hop          1000
Pop              1000
Folk             1000
Experimental     1000
Rock             1000
International    1000
Electronic       1000
Instrumental     1000
Name: count, dtype: int64

Audio files found: 8000
Audio files missing: 0

Final usable tracks: 8000

Final genre distribution:
genre
Hip-Hop          1000
Pop              1000
Folk             1000
Experimental     1000
Rock             1000
International    1000
Electronic       1000
Instrumental     1000
Name: count, dtype: int64

Saved: d:\ML-clgProject\Epochs-Music-Genre-Classifier\data\processed\fma_track_metadata.csv


In [8]:
def fma_audio_path(track_id):
    track_id = int(track_id)
    return AUDIO_DIR / f"{track_id // 1000:03d}" / f"{track_id:06d}.mp3"

sample = track_meta.dropna(subset=["genre"]).head(10).copy()
sample["audio_path"] = sample["track_id"].map(fma_audio_path)
sample["exists"] = sample["audio_path"].map(Path.exists)
sample[["track_id","genre","artist_id","audio_path","exists"]]


,track_id,genre,artist_id,audio_path,exists
track_id,,,,,
2,2,Hip-Hop,1,d:\ML-clgProject\Epochs-Music-Genre-Classifier...,True
5,5,Hip-Hop,1,d:\ML-clgProject\Epochs-Music-Genre-Classifier...,True
10,10,Pop,6,d:\ML-clgProject\Epochs-Music-Genre-Classifier...,True
140,140,Folk,54,d:\ML-clgProject\Epochs-Music-Genre-Classifier...,True
141,141,Folk,54,d:\ML-clgProject\Epochs-Music-Genre-Classifier...,True
148,148,Experimental,57,d:\ML-clgProject\Epochs-Music-Genre-Classifier...,True
182,182,Rock,64,d:\ML-clgProject\Epochs-Music-Genre-Classifier...,True
190,190,Folk,127,d:\ML-clgProject\Epochs-Music-Genre-Classifier...,True
193,193,Folk,127,d:\ML-clgProject\Epochs-Music-Genre-Classifier...,True


In [9]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
out = processed_dir / "fma_track_metadata.csv"
track_meta.to_csv(out, index=False)
print("Saved:", out)


Saved: d:\ML-clgProject\Epochs-Music-Genre-Classifier\data\processed\fma_track_metadata.csv
